In [1]:
import numpy as np
import math

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ============================================================
# WEEK 11 — FUNCTION 2 (CLUSTER-AWARE GP-BASED LOCAL MAXIMISATION)
# What changed vs Week 10 (Module 22: Clustering lens):
#  - Adds KMeans clustering over historical X to find natural groupings
#  - Targets the "promising" cluster (highest y_max, tie-break by y_mean)
#  - Candidate generation now includes:
#      (a) around x_best (local exploitation)
#      (b) around best-cluster centroid (centroid trend tightening)
#      (c) along boundary between top-2 cluster centroids (boundary tightening)
#      (d) global exploration (coverage)
#  - Scoring adds a small cluster-affinity term (prefer close to promising centroid)
#  - Still uses EI + UCB + novelty + min-distance filter for transparency
#  - Deterministic seed for reproducibility
#  - Output x_next rounded to <= 6 decimals
# ============================================================

# ----------------------------
# 1) INPUT DATA (your history)
# ----------------------------
X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],   # will be clipped to 1.0
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],   # duplicate
    [0.99947700, 0.02152800],
    [0.00757800, 0.97735900],
    [1.00000000, 0.98231600],
    [0.69158300, 0.48913100]
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439,
    0.15170334833240093, 0.003163976344064868, 0.4846853396945425
], dtype=float)

# Bounds enforcement for black-box
X = np.clip(X, 0.0, 1.0)

# -------------------------------------------
# 2) DEDUPE (average y for identical points)
# -------------------------------------------
def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)
    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())
    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# -------------------------------------------
# 3) GP model (interpretable uncertainty)
# -------------------------------------------
def make_gp(seed=2026):
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(1e-2, 1e1),
        nu=2.5
    ) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=12,
        random_state=seed
    )

def gp_cv_mse(X, y, seed=2026):
    kf = KFold(n_splits=min(5, len(y)), shuffle=True, random_state=seed)
    mses = []
    for tr, te in kf.split(X):
        gp = make_gp(seed=seed)
        gp.fit(X[tr], y[tr])
        mu, _ = gp.predict(X[te], return_std=True)
        mses.append(mean_squared_error(y[te], mu))
    return float(np.mean(mses))

cv_mse = gp_cv_mse(X, y, seed=2026)

gp = make_gp(seed=2026)
gp.fit(X, y)

# -------------------------------------------
# 4) Acquisition functions (transparent)
# -------------------------------------------
def normal_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)

def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))

def expected_improvement(mu, std, y_best, xi=0.001):
    std = np.maximum(std, 1e-12)
    z = (mu - y_best - xi) / std
    return (mu - y_best - xi) * normal_cdf(z) + std * normal_pdf(z)

# -------------------------------------------
# 5) Clustering lens (Module 22)
#    Find natural groupings in the sampled space
# -------------------------------------------
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

# Simple, explainable choice: k=4 when data is >=12 else k=3
k = 4 if len(X) >= 12 else 3
kmeans = KMeans(n_clusters=k, random_state=2026, n_init=30)
labels = kmeans.fit_predict(Xz)

centroids = scaler.inverse_transform(kmeans.cluster_centers_)

# Rank clusters by best observed performance, then by average performance
cluster_rows = []
for c in range(k):
    idx = np.where(labels == c)[0]
    yc = y[idx]
    cluster_rows.append({
        "cluster": c,
        "n": int(len(idx)),
        "y_mean": float(np.mean(yc)),
        "y_max": float(np.max(yc)),
        "centroid": centroids[c]
    })

cluster_rows_sorted = sorted(
    cluster_rows,
    key=lambda r: (r["y_max"], r["y_mean"]),
    reverse=True
)

best_cluster = cluster_rows_sorted[0]["cluster"]
second_cluster = cluster_rows_sorted[1]["cluster"] if k >= 2 else best_cluster

c0 = cluster_rows_sorted[0]["centroid"]
c1 = cluster_rows_sorted[1]["centroid"] if k >= 2 else c0

# -------------------------------------------
# 6) Candidate generation
#    (global + x_best local + centroid tightening + boundary tightening)
# -------------------------------------------
rng = np.random.default_rng(2026)

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])
n = len(y)

# Trust region shrinks as data grows (explainable heuristic)
tr_sigma = float(np.clip(0.10 / np.sqrt(max(n, 1)), 0.02, 0.05))

# Centroid tightening radius (small, stable)
cent_sigma = 0.035

N_global   = 20000
N_local    = 20000
N_centroid = 20000
N_boundary = 12000

X_global = rng.uniform(0.0, 1.0, size=(N_global, 2))
X_local  = np.clip(rng.normal(loc=x_best, scale=tr_sigma, size=(N_local, 2)), 0.0, 1.0)

# Cluster centroid tightening (targets promising region)
X_cent = np.clip(rng.normal(loc=c0, scale=cent_sigma, size=(N_centroid, 2)), 0.0, 1.0)

# Boundary tightening between top-2 clusters (line + small perpendicular noise)
t = rng.uniform(0.0, 1.0, size=(N_boundary, 1))
line = c0 + t * (c1 - c0)

v = (c1 - c0)
v_norm = np.linalg.norm(v) + 1e-12
v = v / v_norm
perp = np.array([-v[1], v[0]])
noise = rng.normal(0.0, 0.02, size=(N_boundary, 1)) * perp

X_bound = np.clip(line + noise, 0.0, 1.0)

Xcand = np.vstack([X_global, X_local, X_cent, X_bound])

# -------------------------------------------
# 7) Score candidates: EI + UCB + novelty + cluster affinity
# -------------------------------------------
mu, std = gp.predict(Xcand, return_std=True)

xi = 0.001
ei = expected_improvement(mu, std, y_best, xi=xi)

kappa = 2.0
ucb = mu + kappa * std

# Distance filter: avoid repeats
dists = np.sqrt(((Xcand[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
min_dist = dists.min(axis=1)

min_sep = float(np.clip(0.10 / np.sqrt(max(n, 1)), 0.01, 0.025))
valid = min_dist >= min_sep

def zscore(v):
    return (v - v.mean()) / (v.std() + 1e-12)

ei_z  = zscore(ei)
ucb_z = zscore(ucb)
nov_z = zscore(min_dist)

# Cluster affinity: prefer closeness to best-cluster centroid (clustering cue)
dist_to_c0 = np.linalg.norm(Xcand - c0, axis=1)
cluster_aff = -zscore(dist_to_c0)  # higher is better (closer to centroid)

# Week 11 mix (clustering-aligned):
#  - EI still dominates (optimisation)
#  - UCB keeps exploration alive
#  - Novelty prevents collapse
#  - Cluster affinity focuses search within promising “group”
score = 0.55 * ei_z + 0.25 * ucb_z + 0.10 * nov_z + 0.10 * cluster_aff

score_masked = np.where(valid, score, -np.inf)
best_idx = int(np.argmax(score_masked))

x_next = np.round(Xcand[best_idx], 6)

# -------------------------------------------
# 8) Transparency prints
# -------------------------------------------
topk = 5
top_idx = np.argsort(score_masked)[-topk:][::-1]

print("================================================")
print("WEEK 11 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)")
print("================================================")
print(f"GP kernel learned: {gp.kernel_}")
print(f"CV MSE (sanity check): {cv_mse:.6f}")
print(f"x_best = [{x_best[0]:.6f}, {x_best[1]:.6f}], y_best = {y_best:.6f}")
print(f"trust_sigma = {tr_sigma:.6f}, min_sep = {min_sep:.6f}")
print("------------------------------------------------")
print("Cluster summary (k-means on X):")
for r in cluster_rows_sorted:
    cx, cy = r["centroid"]
    print(
        f"  cluster={r['cluster']}  n={r['n']}  "
        f"y_mean={r['y_mean']:.6f}  y_max={r['y_max']:.6f}  "
        f"centroid=[{cx:.6f},{cy:.6f}]"
    )
print("------------------------------------------------")
print(f"Targeted cluster (best by y_max, then y_mean): cluster={best_cluster}")
print(f"Centroid cue: centroid=[{c0[0]:.6f}, {c0[1]:.6f}] (tightening around it)")
print(f"Boundary cue: probing between centroid0 and centroid1")
print("------------------------------------------------")
print(f"x_next = [{x_next[0]:.6f}, {x_next[1]:.6f}]")
print(f"mu(x_next)        = {mu[best_idx]:.6f}")
print(f"std(x_next)       = {std[best_idx]:.6f}")
print(f"EI(x_next)        = {ei[best_idx]:.6f}")
print(f"UCB(x_next)       = {ucb[best_idx]:.6f}")
print(f"min_dist(x_next)  = {min_dist[best_idx]:.6f}")
print(f"dist_to_centroid  = {dist_to_c0[best_idx]:.6f}")
print("score_parts = EI_w=0.55, UCB_w=0.25, nov_w=0.10, clust_w=0.10")
print("------------------------------------------------")
print("Top candidates (for transparency):")
for rank, i in enumerate(top_idx, start=1):
    x = Xcand[i]
    print(
        f"{rank}) x=[{x[0]:.6f},{x[1]:.6f}]  "
        f"mu={mu[i]:.6f} std={std[i]:.6f} EI={ei[i]:.6f} "
        f"UCB={ucb[i]:.6f} min_dist={min_dist[i]:.6f} "
        f"distC={dist_to_c0[i]:.6f} score={score_masked[i]:.6f}"
    )


C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for

WEEK 11 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)
GP kernel learned: 0.849**2 * Matern(length_scale=[0.0385, 10], nu=2.5) + WhiteKernel(noise_level=0.0742)
CV MSE (sanity check): 0.020842
x_best = [0.683406, 0.063769], y_best = 0.629731
trust_sigma = 0.022942, min_sep = 0.022942
------------------------------------------------
Cluster summary (k-means on X):
  cluster=0  n=6  y_mean=0.275900  y_max=0.629731  centroid=[0.778463,0.072088]
  cluster=1  n=7  y_mean=0.256392  y_max=0.611205  centroid=[0.803080,0.808533]
  cluster=2  n=5  y_mean=0.083770  y_max=0.244619  centroid=[0.343182,0.313409]
  cluster=3  n=1  y_mean=0.151703  y_max=0.151703  centroid=[0.007578,0.977359]
------------------------------------------------
Targeted cluster (best by y_max, then y_mean): cluster=0
Centroid cue: centroid=[0.778463, 0.072088] (tightening around it)
Boundary cue: probing between centroid0 and centroid1
------------------------------------------------
x_next = [0.682655, 0.342687]
mu(x_next) 